In [30]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest, f_classif
import pandas as pd
import numpy as np

# Data Cleaning and Preprocessing

In [2]:
df = pd.read_csv("pima_data - Copy.csv")

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,192,66,35,64,31.3,1.668,42,1
1,3,122,56,44,269,42.1,0.562,41,0
2,12,126,99,48,163,18.7,0.810,46,1
3,14,108,103,27,290,26.6,1.587,60,1
4,10,178,74,48,186,34.6,1.539,41,1


In [3]:
# No null values, all numerical types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
df.shape

(768, 9)

In [5]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,6.875000,132.734375,78.796875,33.065104,154.665365,34.554818,1.260259,50.032552,0.769531
std,4.364429,37.549234,23.548344,15.028818,81.667825,9.019027,0.702278,16.981629,0.421407
min,0.000000,70.000000,40.000000,7.000000,15.000000,18.100000,0.100000,21.000000,0.000000
25%,3.000000,102.000000,59.000000,20.000000,82.000000,27.250000,0.642000,36.000000,1.000000
50%,7.000000,131.000000,77.000000,32.000000,155.000000,34.950000,1.234000,49.000000,1.000000
75%,11.000000,166.000000,99.250000,46.000000,224.000000,42.100000,1.883500,65.000000,1.000000
max,14.000000,199.000000,119.000000,59.000000,299.000000,49.900000,2.494000,79.000000,1.000000


In [6]:
# Checking for invalid zeros in any columns using 
# Code from ChatGPT, Prompt: "How to check for zeros in a pandas dataframe columns?"
# zeros in Pregnancies and Outcome columns make sense so I'll keep them
(df == 0).any()

Pregnancies                  True
Glucose                     False
BloodPressure               False
SkinThickness               False
Insulin                     False
BMI                         False
DiabetesPedigreeFunction    False
Age                         False
Outcome                      True
dtype: bool

In [7]:
X = df.drop(["Outcome"], axis=1)
y = df["Outcome"]

X.shape, y.shape

((768, 8), (768,))

In [8]:
df.corr()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
Pregnancies,1.000000,0.035948,-0.047197,0.054090,0.054655,0.010296,-0.018967,-0.015566,0.328125
Glucose,0.035948,1.000000,-0.045341,0.018130,0.033689,0.039485,-0.014549,0.027108,0.301894
BloodPressure,-0.047197,-0.045341,1.000000,0.001117,-0.075879,0.089671,-0.032142,-0.012630,-0.005512
SkinThickness,0.054090,0.018130,0.001117,1.000000,-0.009482,0.004758,0.076386,-0.000586,-0.000510
Insulin,0.054655,0.033689,-0.075879,-0.009482,1.000000,-0.017999,0.062219,0.036972,-0.003418
BMI,0.010296,0.039485,0.089671,0.004758,-0.017999,1.000000,-0.032092,0.021476,0.388904
DiabetesPedigreeFunction,-0.018967,-0.014549,-0.032142,0.076386,0.062219,-0.032092,1.000000,0.034891,0.016674
Age,-0.015566,0.027108,-0.012630,-0.000586,0.036972,0.021476,0.034891,1.000000,0.348303
Outcome,0.328125,0.301894,-0.005512,-0.000510,-0.003418,0.388904,0.016674,0.348303,1.000000


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train.shape, y_train.shape, X_test.shape, y_test.shape

((614, 8), (614,), (154, 8), (154,))

### Scaling the features using pipelines

In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
X_train_scaled.shape, X_test_scaled.shape

((614, 8), (154, 8))

In [15]:
rf_model = RandomForestClassifier(random_state=42)

lr_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=10000, random_state=42))
])

svm_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", SVC(probability=True, random_state=42))
])

knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

### Defining hyperparameter grids

In [17]:
# Note: Use 'model__C' instead of 'C' because the models are nested inside a Pipeline.
# The 'model__' prefix tells the grid search to look specifically at the step named "model". 
# (Source: AI Gemini)
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15]
}
lr_param_grid = {
    'model__C': [0.1, 1, 10]
}
svm_param_grid = {
    'model__kernel': ['linear', 'rbf'],
    'model__C': [0.1, 1, 10],
    'model__gamma': [0.01, 0.1, 1]
}
knn_param_grid = {
    'model__n_neighbors': [3, 5, 7, 9, 11]
}

# Running Grid search CV for each model

### A. Random Forest

In [26]:
# cv=5, 5 fold cross-validation
rf_gs = GridSearchCV(
rf_model,
rf_param_grid,
cv=5,
scoring='accuracy'
)
rf_gs.fit(X_train, y_train)
print("Random Forest best params:", rf_gs.best_params_)
# round: round numbers in python - below is round to 3:
print("Random Forest best CV score:", round(rf_gs.best_score_, 3))

Random Forest best params: {'max_depth': 10, 'n_estimators': 200}
Random Forest best CV score: 0.977


### B. Logistic Regression

In [27]:
lr_gs = GridSearchCV(
lr_pipeline,
lr_param_grid,
cv=5,
scoring='accuracy'
)
lr_gs.fit(X_train, y_train)
print("Logistic Regression best params:", lr_gs.best_params_)
print("Logistic Regression best CV score:", round(lr_gs.best_score_, 3))

Logistic Regression best params: {'model__C': 0.1}
Logistic Regression best CV score: 0.886


### C. Support Vector Machines

In [28]:
svm_gs = GridSearchCV(
svm_pipeline,
svm_param_grid,
cv=5,
scoring='accuracy'
)
svm_gs.fit(X_train, y_train)
print("SVM best params:", svm_gs.best_params_)
print("SVM best CV score:", round(svm_gs.best_score_, 3))

SVM best params: {'model__C': 10, 'model__gamma': 0.1, 'model__kernel': 'rbf'}
SVM best CV score: 0.886


### D. K Nearest Neighbours

In [29]:
knn_gs = GridSearchCV(
knn_pipeline,
knn_param_grid,
cv=5,
scoring='accuracy'
)
knn_gs.fit(X_train, y_train)
print("KNN best params:", knn_gs.best_params_)
print("KNN best CV score:", round(knn_gs.best_score_, 3))

KNN best params: {'model__n_neighbors': 11}
KNN best CV score: 0.876


# Feature Selection

In [ ]:
selector = SelectKBest(score_func=f_classif, k=5)

X_train_kbest = selector.fit_transform(X_train, y_train)
X_test_kbest = selector.transform(X_test)